**Student name:** Amer Surur M Al-Harbi


# Information Retrieval — Term 481 — Assignment 2
## Evaluation, Ranked Retrieval (VSM, BM25, Language Models) & Query Expansion


### Instructions
- Work through each exercise in order. Each exercise has a **"Your Code"** cell for you to complete.
- Every code cell ends with a short **test** block — do not modify it. Your output should match the expected output shown below the cell (small floating-point differences in the last digit are acceptable).
- Use **only the Python standard library** (`math`, `re`, `collections`). Do **not** use PyTerrier, scikit-learn, or NumPy for the exercises — the goal is to implement what PyTerrier did for us in the labs (`TF_IDF`, `BM25`, `Hiemstra_LM`, `RM3`, `pt.Evaluate`) from scratch.
- Unless stated otherwise, `log` means **log base 10** (as in the slides), and ties in a ranking are broken by **document ID order** (`d1` before `d2`, ...).

| Part | Topic | Points |
|---|---|---|
| 1 | Evaluation of ranked retrieval | 25 |
| 2 | Vector Space Model (tf-idf & cosine) | 25 |
| 3 | Okapi BM25 | 10 |
| 4 | Language Models for IR | 20 |
| 5 | Query Expansion & Relevance Feedback | 20 |
| | **Total** | **100** |

---
## Part 1 — Evaluation of Ranked Retrieval

Recall from Topic 3 and the evaluation examples:

- **Precision** $= \dfrac{\#\text{relevant retrieved}}{\#\text{retrieved}}$, **Recall** $= \dfrac{\#\text{relevant retrieved}}{\#\text{relevant in collection}}$
- $F_\beta = \dfrac{(1+\beta^2)\,P\,R}{\beta^2 P + R}$
- **P@k** = precision of the top *k* results.
- **Average Precision (AP)** = sum of the precision values at the rank of each relevant document retrieved, divided by the **total number of relevant documents** (relevant documents that are never retrieved contribute 0). **MAP** = mean of AP over queries.

A retrieval system returned ranked lists of **40 documents** for two queries:

- **Q1:** relevant documents at ranks **1, 4, 8, 10, 15, 30** — there are **8** relevant documents in the whole collection.
- **Q2:** relevant documents at ranks **2, 3, 5, 20** — there are **5** relevant documents in the whole collection.

### Exercise 1.1 — Compute by hand (3 points)

Before writing any code, fill in the table (fractions are fine, e.g. `4/10`). For *"precision when recall first reaches 50%"*, find the **first rank** at which recall becomes $\ge 0.5$ and report the precision at that rank.

| Measure | Q1 | Q2 |
|---|---|---|
| Precision (whole list of 40) | ? | ? |
| Recall (whole list of 40) | ? | ? |
| $F_{0.5}$ | ? | ? |
| P@10 | ? | ? |
| Precision when recall first reaches 50% | ? | ? |
| Average Precision | ? | ? |
| MAP of the system | ? | |

**Answer (Exercise 1.1):**

| Measure | Q1 | Q2 |
|---|---|---|
| Precision (whole list of 40) | 6/40 = 0.15 | 4/40 = 0.10 |
| Recall (whole list of 40) | 6/8 = 0.75 | 4/5 = 0.80 |
| $F_{0.5}$ | 1.25·0.15·0.75 / (0.25·0.15+0.75) = 0.1786 | 1.25·0.10·0.80 / (0.25·0.10+0.80) = 0.1212 |
| P@10 | 4/10 = 0.40 | 3/10 = 0.30 |
| Precision when recall first reaches 50% | recall ≥ 0.5 needs 4 of 8 relevant → rank 10 → 4/10 = 0.40 | needs 3 of 5 (2.5 → 3) → rank 5 → 3/5 = 0.60 |
| Average Precision | (1/1 + 2/4 + 3/8 + 4/10 + 5/15 + 6/30) / 8 = 0.3510 | (1/2 + 2/3 + 3/5 + 4/20) / 5 = 0.3933 |
| MAP of the system | (0.3510 + 0.3933) / 2 = **0.3722** | |


In [1]:
import math
import re
from collections import Counter, defaultdict

runs = {
    "Q1": {"n_retrieved": 40, "rel_ranks": [1, 4, 8, 10, 15, 30], "n_relevant": 8},
    "Q2": {"n_retrieved": 40, "rel_ranks": [2, 3, 5, 20],         "n_relevant": 5},
}

def ranks_to_binary(rel_ranks, n_retrieved):
    # Helper (given): converts relevant ranks into a 0/1 relevance list, e.g. [1, 0, 0, 1, ...]
    return [1 if r in set(rel_ranks) else 0 for r in range(1, n_retrieved + 1)]

### Exercise 1.2 — Set-based and rank-based measures (6 points)

Implement the following functions. `rels` is a 0/1 list in rank order (index 0 = rank 1) and `n_relevant` is the number of relevant documents in the collection.

- `precision(rels)`, `recall(rels, n_relevant)`
- `f_beta(p, r, beta)` (return `0.0` if both `p` and `r` are 0)
- `precision_at_k(rels, k)`
- `precision_at_recall(rels, n_relevant, target)` → precision at the **first** rank where recall $\ge$ `target`.

In [2]:
def precision(rels):
    return sum(rels) / len(rels) if rels else 0.0

def recall(rels, n_relevant):
    return sum(rels) / n_relevant if n_relevant else 0.0

def f_beta(p, r, beta=1.0):
    if p == 0 and r == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * p * r / (b2 * p + r)

def precision_at_k(rels, k):
    return sum(rels[:k]) / k

def precision_at_recall(rels, n_relevant, target):
    hits = 0
    for rank, rel in enumerate(rels, start=1):
        hits += rel
        if n_relevant and hits / n_relevant >= target and hits > 0:
            return hits / rank
    return 0.0

# ---- Test (do not modify) ----
for q, info in runs.items():
    rels = ranks_to_binary(info["rel_ranks"], info["n_retrieved"])
    P, R = precision(rels), recall(rels, info["n_relevant"])
    print(f"{q}: P={P:.4f}  R={R:.4f}  F0.5={f_beta(P, R, 0.5):.4f}  "
          f"P@10={precision_at_k(rels, 10):.4f}  P@R50%={precision_at_recall(rels, info['n_relevant'], 0.5):.4f}")

Q1: P=0.1500  R=0.7500  F0.5=0.1786  P@10=0.4000  P@R50%=0.4000
Q2: P=0.1000  R=0.8000  F0.5=0.1212  P@10=0.3000  P@R50%=0.6000


In [3]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 1.3 — Average Precision & MAP (5 points)

Implement `average_precision(rels, n_relevant)` and `mean_average_precision(list_of_(rels, n_relevant))`.

*(Compare with the slide: $AP_1 = (\frac{1}{5}+\frac{2}{7}+\frac{3}{9}+\frac{4}{10}+\frac{5}{22})/6$.)*

In [4]:
def average_precision(rels, n_relevant):
    hits, total = 0, 0.0
    for rank, rel in enumerate(rels, start=1):
        if rel:
            hits += 1
            total += hits / rank          # precision at this relevant rank
    return total / n_relevant if n_relevant else 0.0

def mean_average_precision(runs_list):
    aps = [average_precision(rels, n_rel) for rels, n_rel in runs_list]
    return sum(aps) / len(aps) if aps else 0.0

# ---- Test (do not modify) ----
runs_list = []
for q, info in runs.items():
    rels = ranks_to_binary(info["rel_ranks"], info["n_retrieved"])
    runs_list.append((rels, info["n_relevant"]))
    print(f"AP({q}) = {average_precision(rels, info['n_relevant']):.4f}")
print(f"MAP     = {mean_average_precision(runs_list):.4f}")

AP(Q1) = 0.3510
AP(Q2) = 0.3933
MAP     = 0.3722


In [5]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 1.4 — Graded relevance: nDCG (5 points)

A query has **8** relevant documents in the collection: **2 perfect (P)**, **2 excellent (E)** and **4 good (G)**; all others are non-relevant (N). The system returned (top of the ranking on the left):

```
N  P  G  E  N  G  P  N  N  G
```

Use gains **P = 3, E = 2, G = 1, N = 0** and the DCG formula used in the evaluation examples:

$$DCG_k = rel_1 + \sum_{i=2}^{k} \frac{rel_i}{\log_2 i}$$

$nDCG_k = DCG_k / IDCG_k$, where the **ideal** ranking sorts **all relevant documents in the collection** (not just the retrieved ones) by decreasing gain.

Implement `dcg(gains, k)` and `ndcg(gains, all_relevant_gains, k)`, then report DCG@10, IDCG@10, nDCG@10 and nDCG@5.

In [6]:
GAIN = {"P": 3, "E": 2, "G": 1, "N": 0}
ranking = "N P G E N G P N N G".split()
collection_judgments = ["P"] * 2 + ["E"] * 2 + ["G"] * 4   # all relevant docs in the collection

def dcg(gains, k):
    total = 0.0
    for i, g in enumerate(gains[:k], start=1):
        total += g if i == 1 else g / math.log2(i)
    return total

def ndcg(gains, all_relevant_gains, k):
    ideal = dcg(sorted(all_relevant_gains, reverse=True), k)
    return dcg(gains, k) / ideal if ideal > 0 else 0.0

# ---- Test (do not modify) ----
gains = [GAIN[x] for x in ranking]
all_gains = [GAIN[x] for x in collection_judgments]
print(f"DCG@10  = {dcg(gains, 10):.4f}")
print(f"IDCG@10 = {dcg(sorted(all_gains, reverse=True), 10):.4f}")
print(f"nDCG@10 = {ndcg(gains, all_gains, 10):.4f}")
print(f"nDCG@5  = {ndcg(gains, all_gains, 5):.4f}")

DCG@10  = 6.3874
IDCG@10 = 9.7689
nDCG@10 = 0.6539
nDCG@5  = 0.5327


In [7]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 1.5 — Pooling (6 points)

In a TREC-style evaluation, 3 teams submitted ranked lists (up to 12 documents) for a query $Q$. With exhaustive judgments, $Q$ has **9** relevant documents: **A, B, C, D, E, F, G, H, I**.

| Rank | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| T1 | X | B | K | Y | D | E | M | H | I | W | L | Z |
| T2 | K | M | F | X | B | N | A | P | G | W | H | Z |
| T3 | C | G | Y | L | Q | B | K | I | R | D | X | M |

The judging **pool** is built from **only the top 4 documents** of each submitted list. Only pooled documents get judged; anything outside the pool is assumed **non-relevant**.

1. Implement `build_pool(runs, depth)` → the set of pooled documents.
2. Implement `pooled_qrels(pool, true_relevant)` → the set of documents judged relevant.
3. For each team, compute AP **(a)** with the pooled judgments, dividing by the number of relevant documents *found in the pool* (as in the slide example), and **(b)** with the true exhaustive judgments (dividing by 9). Re-use `average_precision` from Exercise 1.3.

Then answer in a markdown cell: *does pooling change the ranking of the three teams? Which team is hurt the most by the shallow pool, and why?*

In [8]:
team_runs = {
    "T1": "X B K Y D E M H I W L Z".split(),
    "T2": "K M F X B N A P G W H Z".split(),
    "T3": "C G Y L Q B K I R D X M".split(),
}
true_relevant = set("ABCDEFGHI")

def build_pool(runs, depth):
    pool = set()
    for ranking in runs.values():
        pool.update(ranking[:depth])
    return pool

def pooled_qrels(pool, true_relevant):
    return set(pool) & set(true_relevant)

# ---- Test (do not modify) ----
pool = build_pool(team_runs, 4)
qrels_pool = pooled_qrels(pool, true_relevant)
print("Pool:              ", sorted(pool))
print("Judged relevant:   ", sorted(qrels_pool))
for team, ranking in team_runs.items():
    rels_pool = [1 if d in qrels_pool else 0 for d in ranking]
    rels_true = [1 if d in true_relevant else 0 for d in ranking]
    ap_pool = average_precision(rels_pool, len(qrels_pool))
    ap_true = average_precision(rels_true, len(true_relevant))
    print(f"{team}: AP(pooled) = {ap_pool:.4f}   AP(exhaustive) = {ap_true:.4f}")

Pool:               ['B', 'C', 'F', 'G', 'K', 'L', 'M', 'X', 'Y']
Judged relevant:    ['B', 'C', 'F', 'G']
T1: AP(pooled) = 0.1250   AP(exhaustive) = 0.2728
T2: AP(pooled) = 0.2667   AP(exhaustive) = 0.2290
T3: AP(pooled) = 0.6250   AP(exhaustive) = 0.3889


In [9]:
# The output generated by the code written to answer this question should be as follows:

**Your answer (Exercise 1.5):** Yes, pooling changes the ranking of the teams.

| | exhaustive AP | pooled AP |
|---|---|---|
| T3 | 0.3889 (1st) | 0.6250 (1st) |
| T1 | 0.2728 (2nd) | 0.1250 (**3rd**) |
| T2 | 0.2290 (3rd) | 0.2667 (**2nd**) |

With the true judgments the order is T3 > T1 > T2, but with the pooled judgments T1 and T2 swap: T1 drops from 2nd to last.

**T1 is hurt the most** (AP falls from 0.2728 to 0.1250, i.e. more than half). The pool contains only the top-4 documents of each run = {B, C, F, G, K, L, M, X, Y}, so only 4 relevant documents (B, C, F, G) are ever judged. T1's relevant documents are D, E, H, I (ranks 5, 6, 8, 9) – none of them is in any team's top 4, so they were never judged and are counted as non-relevant; only B (rank 2) survives. T1 is the system that finds *unique* relevant documents deeper in the list, which a shallow pool cannot reward. T3, in contrast, has C and G in its own top 4 (they are in the pool because of T3 itself) and is favoured, and T2 benefits from F and G being at the top of its own list. In addition, dividing by 4 (relevant docs found in the pool) instead of 9 inflates every AP. So shallow pooling is biased toward systems that contributed to the pool with similar top-ranked results and penalises systems that retrieve different relevant documents lower in the ranking (they are treated as non-relevant); pool depth must be large enough, or the pooled collection reused for new systems with care.

---
## Part 2 — Vector Space Model

From Part 2 onwards we use the following collection of 20 short documents (think of them as tweets, like EveTAR in the labs). The preprocessing function is **given**: lower-casing, splitting at non-letters, and stop-word removal (no stemming). **Apply the same `preprocess()` to queries and documents** — exactly as we did with the Arabic queries in Labs 4–6.

In [10]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "of", "to", "in", "on", "for", "and", "it", "its",
    "with", "as", "by", "at", "from", "that", "this", "will", "be", "has", "have", "after", "into",
    "their", "new",
}

def tokenize(text):
    return [t for t in re.split(r"[^a-z]+", text.lower()) if t]

def preprocess(text):
    return [t for t in tokenize(text) if t not in STOPWORDS]

docs = {
    "d1":  "NASA will launch a satellite into orbit to study the climate of the earth",
    "d2":  "A rocket will launch the weather satellite into orbit next week",
    "d3":  "The satellite in orbit sends images of the earth to scientists",
    "d4":  "The company will launch a new phone and a new laptop this month, the phone launch is in May",
    "d5":  "Scientists use satellite images to measure melting ice and climate change on earth",
    "d6":  "The football team won the cup after a late goal",
    "d7":  "Engineers repaired the antenna of the space station in orbit",
    "d8":  "Astronauts on the space station orbit the earth every ninety minutes",
    "d9":  "Fans of the football team celebrated the world cup win",
    "d10": "The rocket carried the satellite to orbit after a successful launch",
    "d11": "The bank will launch a new savings account for students",
    "d12": "Heavy rain and strong wind in the weather forecast for the weekend",
    "d13": "The coffee shop sells tea and cake",
    "d14": "Students prepare for the final exams at the university",
    "d15": "The new phone has a better camera and a bigger battery",
    "d16": "Space rocket engines are tested in the desert",
    "d17": "The team will play the final match at the stadium",
    "d18": "Climate change causes melting ice in the arctic",
    "d19": "The museum opens a new exhibition about ancient history",
    "d20": "Prices of laptops and phones fell this month",
}

doc_tokens = {d: preprocess(t) for d, t in docs.items()}
N = len(doc_tokens)
for d, toks in doc_tokens.items():
    print(f"{d:4}: {toks}")

d1  : ['nasa', 'launch', 'satellite', 'orbit', 'study', 'climate', 'earth']
d2  : ['rocket', 'launch', 'weather', 'satellite', 'orbit', 'next', 'week']
d3  : ['satellite', 'orbit', 'sends', 'images', 'earth', 'scientists']
d4  : ['company', 'launch', 'phone', 'laptop', 'month', 'phone', 'launch', 'may']
d5  : ['scientists', 'use', 'satellite', 'images', 'measure', 'melting', 'ice', 'climate', 'change', 'earth']
d6  : ['football', 'team', 'won', 'cup', 'late', 'goal']
d7  : ['engineers', 'repaired', 'antenna', 'space', 'station', 'orbit']
d8  : ['astronauts', 'space', 'station', 'orbit', 'earth', 'every', 'ninety', 'minutes']
d9  : ['fans', 'football', 'team', 'celebrated', 'world', 'cup', 'win']
d10 : ['rocket', 'carried', 'satellite', 'orbit', 'successful', 'launch']
d11 : ['bank', 'launch', 'savings', 'account', 'students']
d12 : ['heavy', 'rain', 'strong', 'wind', 'weather', 'forecast', 'weekend']
d13 : ['coffee', 'shop', 'sells', 'tea', 'cake']
d14 : ['students', 'prepare', 'final'

### Exercise 2.1 — Jaccard scoring (3 points)

Implement `jaccard(A, B)` for two **sets** and `rank_jaccard(query, doc_tokens)` that returns a list of `(doc_id, score)` sorted by decreasing score (ties by doc order), **excluding documents with score 0**. Test it on the query **`"satellite launch"`**.

Then think about the issues listed on the slide *"Issues With Jaccard for Scoring"*: notice that `d11` (a bank launching a savings account!) outranks the relevant `d3` and `d5`, and that `d4` gets no credit for mentioning "launch" twice.

In [11]:
def jaccard(A, B):
    A, B = set(A), set(B)
    union = A | B
    return len(A & B) / len(union) if union else 0.0

def rank_jaccard(query, doc_tokens):
    q = set(preprocess(query))
    scored = [(d, jaccard(q, set(toks))) for d, toks in doc_tokens.items()]
    scored = [(d, s) for d, s in scored if s > 0]
    return sorted(scored, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
for d, s in rank_jaccard("satellite launch", doc_tokens):
    print(f"{d:4} {s:.4f}")

d10  0.3333
d1   0.2857
d2   0.2857
d11  0.1667
d3   0.1429
d4   0.1429
d5   0.0909


In [12]:
# The output generated by the code written to answer this question should be as follows:

**Observation (Exercise 2.1):** with Jaccard `d10` (6 distinct terms) ranks first; `d11` (a bank launching a savings account, 5 terms, only 1 match) scores 0.1667 and is ranked above the truly relevant `d3` and `d5` (0.1429, 0.0909) because Jaccard only normalises by union size: it ignores term frequency (`d4` mentions *launch* twice but gets the same credit as a single mention), and it treats the rare, informative term *satellite* and the common term *launch* as equally important (no idf). Also long relevant documents (`d5` has 9 distinct terms) are penalised by the union size.

### Exercise 2.2 — Term statistics: tf, df, idf (5 points)

Implement:
- `term_frequencies(doc_tokens)` → `{doc_id: Counter(term -> tf)}`
- `document_frequencies(doc_tokens)` → `Counter(term -> df)`
- `idf(df, N)` → $\log_{10}(N/df)$
- `log_tf(tf)` → $1+\log_{10}(tf)$ if $tf>0$ else $0$

In [13]:
def term_frequencies(doc_tokens):
    return {d: Counter(toks) for d, toks in doc_tokens.items()}

def document_frequencies(doc_tokens):
    df = Counter()
    for toks in doc_tokens.values():
        df.update(set(toks))
    return df

def idf(df_t, N):
    return math.log10(N / df_t)

def log_tf(tf):
    return 1 + math.log10(tf) if tf > 0 else 0.0

# ---- Test (do not modify) ----
TF = term_frequencies(doc_tokens)
DF = document_frequencies(doc_tokens)
for t in ["satellite", "launch", "orbit", "earth", "phone", "space", "rocket"]:
    print(f"{t:10} df={DF[t]}  idf={idf(DF[t], N):.4f}  tf in docs: "
          + str({d: TF[d][t] for d in TF if TF[d][t] > 0}))
print("log_tf:", [round(log_tf(x), 4) for x in [0, 1, 2, 10, 1000]])

satellite  df=5  idf=0.6021  tf in docs: {'d1': 1, 'd2': 1, 'd3': 1, 'd5': 1, 'd10': 1}
launch     df=5  idf=0.6021  tf in docs: {'d1': 1, 'd2': 1, 'd4': 2, 'd10': 1, 'd11': 1}
orbit      df=6  idf=0.5229  tf in docs: {'d1': 1, 'd2': 1, 'd3': 1, 'd7': 1, 'd8': 1, 'd10': 1}
earth      df=4  idf=0.6990  tf in docs: {'d1': 1, 'd3': 1, 'd5': 1, 'd8': 1}
phone      df=2  idf=1.0000  tf in docs: {'d4': 2, 'd15': 1}
space      df=3  idf=0.8239  tf in docs: {'d7': 1, 'd8': 1, 'd16': 1}
rocket     df=3  idf=0.8239  tf in docs: {'d2': 1, 'd10': 1, 'd16': 1}
log_tf: [0.0, 1.0, 1.301, 2.0, 4.0]


In [14]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 2.3 — tf-idf overlap score (5 points)

Implement `rank_tfidf(query, ...)` where (slide *"tf.idf Term Weighting"*):

$$score(q,d) = \sum_{t \in q \cap d} \left(1+\log_{10} tf_{t,d}\right) \times \log_{10}\frac{N}{df_t}$$

Return `(doc_id, score)` pairs sorted by decreasing score, excluding zeros. Test on **`"satellite launch"`** and on **`"satellite images of the earth"`**.

In [15]:
def rank_tfidf(query, TF, DF, N):
    q_terms = set(preprocess(query))
    scored = []
    for d, tf in TF.items():
        s = sum(log_tf(tf[t]) * idf(DF[t], N) for t in q_terms if tf[t] > 0)
        if s > 0:
            scored.append((d, s))
    return sorted(scored, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
for q in ["satellite launch", "satellite images of the earth"]:
    print(f"Query: {q!r}")
    for d, s in rank_tfidf(q, TF, DF, N):
        print(f"   {d:4} {s:.4f}")

Query: 'satellite launch'
   d1   1.2041
   d2   1.2041
   d10  1.2041
   d4   0.7833
   d3   0.6021
   d5   0.6021
   d11  0.6021
Query: 'satellite images of the earth'
   d3   2.3010
   d5   2.3010
   d1   1.3010
   d8   0.6990
   d2   0.6021
   d10  0.6021


In [16]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 2.4 — Cosine similarity with SMART `lnc.ltc` (7 points)

Using the SMART notation from the slide *"Variants of tf-idf Weighting"*:

- **Documents — `lnc`:** weight $= 1+\log_{10} tf$ (l), **no** idf (n), cosine-normalized (c).
- **Queries — `ltc`:** weight $= (1+\log_{10} tf) \times \log_{10}(N/df)$ (l, t), cosine-normalized (c).

Implement:
- `l2_normalize(vec)` for a sparse vector `{term: weight}`
- `doc_vector_lnc(tf)` and `query_vector_ltc(query, DF, N)` (ignore query terms that do not occur in the collection)
- `cosine(v1, v2)` for two **already-normalized** sparse vectors (just the dot product)
- `rank_cosine(query, ...)` → sorted `(doc_id, score)` list excluding zeros.

Test on **`"satellite launch"`**.

In [17]:
def l2_normalize(vec):
    norm = math.sqrt(sum(w * w for w in vec.values()))
    return {t: w / norm for t, w in vec.items()} if norm > 0 else dict(vec)

def doc_vector_lnc(tf):
    return l2_normalize({t: log_tf(c) for t, c in tf.items() if c > 0})

def query_vector_ltc(query, DF, N):
    qtf = Counter(preprocess(query))
    vec = {t: log_tf(c) * idf(DF[t], N) for t, c in qtf.items() if DF.get(t, 0) > 0}
    return l2_normalize(vec)

def cosine(v1, v2):
    if len(v1) > len(v2):
        v1, v2 = v2, v1
    return sum(w * v2.get(t, 0.0) for t, w in v1.items())

def rank_cosine(query, TF, DF, N):
    q = query_vector_ltc(query, DF, N)
    scored = []
    for d, tf in TF.items():
        s = cosine(q, doc_vector_lnc(tf))
        if s > 0:
            scored.append((d, s))
    return sorted(scored, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
print("query vector:", {t: round(w, 4) for t, w in query_vector_ltc("satellite launch", DF, N).items()})
for d, s in rank_cosine("satellite launch", TF, DF, N):
    print(f"   {d:4} {s:.4f}")

query vector: {'satellite': 0.7071, 'launch': 0.7071}
   d10  0.5774
   d1   0.5345
   d2   0.5345
   d4   0.3385
   d11  0.3162
   d3   0.2887
   d5   0.2236


In [18]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 2.5 — Why cosine and not Euclidean distance? (2 points)

Take document `d10` and create `d10'` by appending `d10` to itself (slide *"Euclidean Distance?"*). Using **raw tf** vectors (no weighting, no normalization) compute the Euclidean distance between `d10` and `d10'`, and the **cosine** between them. Then compute the Euclidean distance between the two **length-normalized** vectors.

In [19]:
def euclidean(v1, v2):
    terms = set(v1) | set(v2)
    return math.sqrt(sum((v1.get(t, 0.0) - v2.get(t, 0.0)) ** 2 for t in terms))

# ---- Test (do not modify) ----
d = dict(Counter(doc_tokens["d10"]))
d_prime = dict(Counter(doc_tokens["d10"] * 2))
print(f"Euclidean(d10, d10')            = {euclidean(d, d_prime):.4f}")
print(f"cosine(d10, d10')               = {cosine(l2_normalize(d), l2_normalize(d_prime)):.4f}")
print(f"Euclidean(norm(d10), norm(d10'))= {euclidean(l2_normalize(d), l2_normalize(d_prime)):.4f}")

Euclidean(d10, d10')            = 2.4495
cosine(d10, d10')               = 1.0000
Euclidean(norm(d10), norm(d10'))= 0.0000


In [20]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 2.6 — Term-at-a-time (TAAT) query processing (3 points)

Ranking by computing a full cosine with **every** document (as in 2.4) is wasteful. Implement the slide's `COSINESCORE(q)` algorithm, which only touches documents in the postings lists of the query terms:

```
1  float Scores[N] = 0
2  Initialize Length[N]
3  for each query term t
4  do calculate w(t,q) and fetch postings list for t
5      for each pair (d, tf(t,d)) in postings list
6      do Scores[d] += wf(t,d) × w(t,q)
7  Read the array Length[d]
8  for each d
9  do Scores[d] = Scores[d] / Length[d]
10 return Top K components of Scores[]
```

1. `build_tf_index(doc_tokens)` → `term -> [(doc_id, tf), ...]` (postings in doc order).
2. `doc_lengths(TF)` → `doc_id -> L2 length of its lnc vector` (computed **offline**).
3. `cosine_score_taat(query, index, lengths, DF, N, K)` — use $wf_{t,d}=1+\log_{10}tf$ and $w_{t,q}$ from `ltc` (already query-normalized), and return the top-K list.

Your scores must be identical to Exercise 2.4.

In [21]:
def build_tf_index(doc_tokens):
    index = defaultdict(list)
    for d, toks in doc_tokens.items():          # docs are visited in doc order
        for t, c in Counter(toks).items():
            index[t].append((d, c))
    return dict(index)

def doc_lengths(TF):
    return {d: math.sqrt(sum(log_tf(c) ** 2 for c in tf.values())) for d, tf in TF.items()}

def cosine_score_taat(query, index, lengths, DF, N, K=10):
    q = query_vector_ltc(query, DF, N)
    scores = defaultdict(float)
    for t, w_tq in q.items():                  # term-at-a-time
        for d, tf in index.get(t, []):
            scores[d] += log_tf(tf) * w_tq
    for d in scores:                           # length-normalise only touched docs
        scores[d] /= lengths[d]
    ranked = sorted(scores.items(), key=lambda x: (-round(x[1], 12), int(x[0][1:])))
    return ranked[:K]

# ---- Test (do not modify) ----
tf_index = build_tf_index(doc_tokens)
lengths = doc_lengths(TF)
print("postings(launch):", tf_index["launch"])
for d, s in cosine_score_taat("satellite launch", tf_index, lengths, DF, N, K=3):
    print(f"   {d:4} {s:.4f}")

postings(launch): [('d1', 1), ('d2', 1), ('d4', 2), ('d10', 1), ('d11', 1)]
   d10  0.5774
   d1   0.5345
   d2   0.5345


In [22]:
# The output generated by the code written to answer this question should be as follows:

---
## Part 3 — Okapi BM25

From the slide *"Okapi BM25 Ranking Function"*:

$$w_{t,d} = \frac{(k_1+1)\,tf_{t,d}}{k_1\left((1-b) + b\,\frac{L_d}{L_{a}}\right) + tf_{t,d}} \times \log_{10}\frac{N - df_t + 0.5}{df_t + 0.5}
\qquad\qquad score(q,d)=\sum_{t\in q\cap d} w_{t,d}$$

where $L_d$ is the length of $d$ (number of tokens **after** preprocessing) and $L_a$ is the average document length in the collection. Use the slide's parameters $k_1 = 2$, $b = 0.75$ as the defaults.

### Exercise 3.1 — Implement BM25 (7 points)

Implement `bm25_term_weight(...)` and `rank_bm25(query, ...)` (returns sorted `(doc_id, score)` pairs excluding documents that contain **no** query term). Test on **`"satellite launch"`**.

In [23]:
def bm25_term_weight(tf, df_t, N, L_d, L_avg, k1=2.0, b=0.75):
    tf_part = (k1 + 1) * tf / (k1 * ((1 - b) + b * L_d / L_avg) + tf)
    idf_part = math.log10((N - df_t + 0.5) / (df_t + 0.5))
    return tf_part * idf_part

def rank_bm25(query, TF, DF, N, doc_len, k1=2.0, b=0.75):
    L_avg = sum(doc_len.values()) / len(doc_len)
    q_terms = set(preprocess(query))
    scored = []
    for d, tf in TF.items():
        matched = [t for t in q_terms if tf[t] > 0]
        if not matched:
            continue
        s = sum(bm25_term_weight(tf[t], DF[t], N, doc_len[d], L_avg, k1, b) for t in matched)
        scored.append((d, s))
    return sorted(scored, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
doc_len = {d: len(toks) for d, toks in doc_tokens.items()}
print("L_avg =", sum(doc_len.values()) / N)
for d, s in rank_bm25("satellite launch", TF, DF, N, doc_len):
    print(f"   {d:4} {s:.4f}")

L_avg = 6.25
   d10  0.9183
   d1   0.8490
   d2   0.8490
   d4   0.6108
   d11  0.5000
   d3   0.4592
   d5   0.3461


In [24]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 3.2 — Effect of the parameters (3 points)

Re-run the query **`"satellite launch"`** with (a) $b=0$ (no length normalization), (b) $b=1$ (full length normalization), and (c) $k_1=0$ (binary tf, keeping $b = 0.75$). Print the three rankings.

In a markdown cell, explain: why does `d4` (the long phone-launch document) move when $b$ changes, and what does $k_1=0$ do to the repeated term "launch" in `d2` and `d4`?

In [25]:
settings = [("b=0", dict(k1=2.0, b=0.0)), ("b=1", dict(k1=2.0, b=1.0)), ("k1=0", dict(k1=0.0, b=0.75))]

# ---- Test (do not modify) ----
for name, params in settings:
    ranking = rank_bm25("satellite launch", TF, DF, N, doc_len, **params)
    print(f"{name:5}: " + ", ".join(f"{d}({s:.4f})" for d, s in ranking))

b=0  : d1(0.8999), d2(0.8999), d10(0.8999), d4(0.6750), d3(0.4500), d5(0.4500), d11(0.4500)
b=1  : d10(0.9246), d1(0.8333), d2(0.8333), d4(0.5921), d11(0.5192), d3(0.4623), d5(0.3214)
k1=0 : d1(0.8999), d2(0.8999), d10(0.8999), d3(0.4500), d4(0.4500), d5(0.4500), d11(0.4500)


In [26]:
# The output generated by the code written to answer this question should be as follows:

**Your answer (Exercise 3.2):**

- **Why `d4` moves with `b`.** `d4` has 8 tokens, longer than the average length of 6.25 (L_d/L_avg = 1.28). With `b=0` there is no length normalisation, so `d4` benefits from its double occurrence of *launch* (tf=2) and scores 0.6750, well above `d3`, `d5`, `d11`. With `b=0.75` its score decreases to 0.6108 and with `b=1` (full normalisation) it falls further to 0.5921, because a long document is penalised – the denominator `k1·(1-b+b·L_d/L_avg)+tf` grows with its length. Short documents (`d11` has 5 tokens, `d10` has 6) move in the opposite direction: `d11` goes 0.4500 → 0.5000 → 0.5192 and overtakes `d3` and `d5`, while `d5` (10 tokens, the longest) falls from 0.4500 to 0.3214.
- **What `k1=0` does.** With `k1=0` the tf part becomes `(0+1)·tf/(0+tf) = 1` for every tf > 0: tf saturates immediately and BM25 becomes a binary (presence/absence) model weighted only by idf; document length no longer matters either (it is multiplied by k1). The repeated *launch* in `d4` (tf=2) counts exactly like one occurrence, so `d4` falls to 0.4500, tying with `d3`, `d5`, `d11` (ties broken by doc ID), and `d2`/`d1`/`d10` score 0.8999 each, which are the same since they match both terms. In `d2`, *launch* occurs only once so it is unaffected by tf saturation (the only change is the removed length normalisation).

---
## Part 4 — Language Models for IR

### Exercise 4.1 — Unigram language models (5 points)

**(a)** A "document" is a bucket of 10 coloured balls (slide *"Unigram Language Model"*):
```
bucket = red red blue green red blue yellow red blue red
```
Implement `mle_model(tokens)` that returns the MLE unigram model $P_{MLE}(w\mid M_d)=\frac{tf_{w,d}}{|d|}$ as a dict, and `sequence_prob(model, seq)` that returns the probability of generating a sequence (independence assumption; unseen words get probability 0). Compute the probabilities of `red blue red yellow` and `red blue purple red`.

**(b)** Two document models are given below (like the slide *"Comparing Language Models"*). Which one is more likely to generate the text `riyadh university research grant`?

In [27]:
bucket = "red red blue green red blue yellow red blue red".split()

M_d1 = {"riyadh": 0.15, "grant": 0.0002, "university": 0.02, "research": 0.0004, "campus": 0.01}
M_d2 = {"riyadh": 0.10, "grant": 0.05,   "university": 0.01, "research": 0.03,   "campus": 0.002}

def mle_model(tokens):
    n = len(tokens)
    return {w: c / n for w, c in Counter(tokens).items()}

def sequence_prob(model, seq):
    p = 1.0
    for w in seq:
        p *= model.get(w, 0.0)
    return p

# ---- Test (do not modify) ----
M = mle_model(bucket)
print("M_d:", M)
print("P(red blue red yellow) =", round(sequence_prob(M, "red blue red yellow".split()), 6))
print("P(red blue purple red) =", round(sequence_prob(M, "red blue purple red".split()), 6))
text = "riyadh university research grant".split()
print(f"P(text|M_d1) = {sequence_prob(M_d1, text):.3e}")
print(f"P(text|M_d2) = {sequence_prob(M_d2, text):.3e}")

M_d: {'red': 0.5, 'blue': 0.3, 'green': 0.1, 'yellow': 0.1}
P(red blue red yellow) = 0.0075
P(red blue purple red) = 0.0
P(text|M_d1) = 2.400e-10
P(text|M_d2) = 1.500e-06


In [28]:
# The output generated by the code written to answer this question should be as follows:

**Answer (Exercise 4.1b):** `M_d2` is much more likely to generate the text: P(text | M_d1) = 0.15·0.02·0.0004·0.0002 = 2.4·10⁻¹⁰ versus P(text | M_d2) = 0.10·0.01·0.03·0.05 = 1.5·10⁻⁶ (about 6000 times larger). Although `M_d1` gives *riyadh* a higher probability, it gives very small probabilities to *research* and *grant*, and since the probabilities are multiplied, the weakest terms dominate the product.

### Exercise 4.2 — Query likelihood with MLE (5 points)

Back to our 10-document collection. Implement `rank_query_likelihood_mle(query, doc_tokens)` that ranks **all** documents (the test prints the top 5) by

$$P(q\mid M_d) = \prod_{q_i \in q} \frac{tf_{q_i,d}}{|d|}$$

Test on **`"satellite orbit earth"`**. How many documents get probability exactly 0 ("veto power")?

In [29]:
def rank_query_likelihood_mle(query, doc_tokens):
    q = preprocess(query)
    res = []
    for d, toks in doc_tokens.items():
        model = mle_model(toks)
        res.append((d, sequence_prob(model, q)))
    return sorted(res, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
res = rank_query_likelihood_mle("satellite orbit earth", doc_tokens)
for d, p in res[:5]:
    print(f"   {d:4} {p:.6f}")
print("   ...")
print("documents with P = 0:", sum(1 for _, p in res if p == 0))

   d3   0.004630
   d1   0.002915
   d2   0.000000
   d4   0.000000
   d5   0.000000
   ...
documents with P = 0: 18


In [30]:
# The output generated by the code written to answer this question should be as follows:

**Answer (Exercise 4.2):** **18 of the 20 documents get probability exactly 0.** Only `d1` and `d3` contain all three query terms (*satellite*, *orbit*, *earth*); every other document is missing at least one of them and, since the MLE query likelihood is a product, a single zero makes the whole product zero ("veto power"). This also makes the ranking almost useless (e.g. `d2`, `d5`, `d8`, `d10` contain two of the three terms but tie with completely unrelated documents at 0), which motivates smoothing.

### Exercise 4.3 — Jelinek–Mercer smoothing (10 points)

Smoothing mixes the document model with the **collection** model (as `Hiemstra_LM` did in Lab 5):

$$P(t\mid d) = \lambda\,\frac{tf_{t,d}}{|d|} + (1-\lambda)\,\frac{cf_t}{|C|}$$

where $cf_t$ is the number of occurrences of $t$ in the whole collection and $|C|$ is the total number of tokens in the collection. Rank by the **log-likelihood** $\sum_{q_i} \log_{10} P(q_i\mid d)$ (to avoid underflow; query terms that never occur in the collection are skipped).

1. Implement `collection_model(doc_tokens)` and `rank_jm(query, doc_tokens, lam)`.
2. Rank all documents for **`"satellite station launch"`** with $\lambda = 0.95$ and $\lambda = 0.15$ (the two values used in Lab 5); the test prints the top 6.

In a markdown cell: compare the two rankings. Where does `d7` (which contains only the rare term *station*) end up in each ranking, and how does this illustrate the lab's remark that *a high λ gives "conjunctive-like" search while a low λ is more disjunctive*?

In [31]:
def collection_model(doc_tokens):
    cf = Counter()
    for toks in doc_tokens.values():
        cf.update(toks)
    total = sum(cf.values())
    return {t: c / total for t, c in cf.items()}      # P(t | C) = cf_t / |C|

def rank_jm(query, doc_tokens, lam):
    p_coll = collection_model(doc_tokens)
    q = [t for t in preprocess(query) if t in p_coll]  # skip terms unseen in the collection
    res = []
    for d, toks in doc_tokens.items():
        tf, L = Counter(toks), len(toks)
        ll = sum(math.log10(lam * tf[t] / L + (1 - lam) * p_coll[t]) for t in q)
        res.append((d, ll))
    return sorted(res, key=lambda x: (-round(x[1], 12), int(x[0][1:])))

# ---- Test (do not modify) ----
for lam in [0.95, 0.15]:
    print(f"lambda = {lam}")
    for d, s in rank_jm("satellite station launch", doc_tokens, lam)[:6]:
        print(f"   {d:4} {s:.4f}")

lambda = 0.95
   d10  -4.6858
   d1   -4.8177
   d2   -4.8177
   d7   -6.1170
   d8   -6.2412
   d4   -6.4158
lambda = 0.15
   d7   -4.2713
   d10  -4.2774
   d1   -4.3287
   d2   -4.3287
   d8   -4.3480
   d4   -4.4412


In [32]:
# The output generated by the code written to answer this question should be as follows:

**Your answer (Exercise 4.3):**

- **λ = 0.95** (mostly the document model): `d10`, `d1`, `d2` – the only documents containing *both* *satellite* and *launch* – are on top; `d7` (only *station*) is 4th with −6.12, far behind (−4.69 … −4.82). Documents matching more query terms are strongly preferred, since a missing term is only rescued by the small collection component 0.05·P(t|C) (a big log penalty), so the model behaves like a *conjunctive* (AND-like) search.
- **λ = 0.15** (mostly the collection model): the log-penalty for a missing term is much smaller, so a single match on a *rare* term is enough. `d7` jumps to **rank 1** (−4.2713), above `d10` (−4.2774), `d1`, `d2`: *station* is rare (df = 2, cf = 2), so tf/|d| = 1/6 for it is a huge boost relative to its collection probability, and the weight of *station* outweighs the other documents matching *satellite* and *launch*, which are more frequent terms. The ranking is more *disjunctive* (OR-like): matching any (rare) query term is rewarded, and the idf-like effect of the collection model dominates.
- This confirms the lab's remark: high λ → conjunctive-like, favours documents containing all query terms; low λ → disjunctive and more idf-driven.

---
## Part 5 — Query Expansion & Relevance Feedback

### Exercise 5.1 — Term association measures for an automatic thesaurus (5 points)

Recall the slide *"Using Term Association Measures"*, where $n_a$, $n_b$ are the numbers of documents containing $a$ and $b$, and $n_{ab}$ the number containing both:

- **Dice:** $\dfrac{2\,n_{ab}}{n_a+n_b}$
- **Mutual information (rank-equivalent form):** $\dfrac{n_{ab}}{n_a \cdot n_b}$

Using the small collection below (preprocessed with the same `preprocess()`), implement `association(a, b, docs_sets, measure)` and `top_associated(term, docs_sets, measure, k)` returning the top-k other terms (ties broken alphabetically). Show the top 4 terms associated with **`cup`** and with **`world`** under each measure.

In a markdown cell: which measure favours **rare** terms, and how does the *"world cup"* example on the slides show the problem of expanding each query term independently?

In [33]:
assoc_docs = [
    "world cup football final tonight",
    "fifa world cup football",
    "football fans watch the world cup",
    "a cup of coffee and a glass of water",
    "drink a cup of tea with milk",
    "world news today",
    "football league results",
    "the champions lifted the cup trophy",
    "coffee cup and tea cup",
    "world football rankings",
]
docs_sets = [set(preprocess(t)) for t in assoc_docs]

def association(a, b, docs_sets, measure="dice"):
    na = sum(1 for s in docs_sets if a in s)
    nb = sum(1 for s in docs_sets if b in s)
    nab = sum(1 for s in docs_sets if a in s and b in s)
    if na == 0 or nb == 0:
        return 0.0
    if measure == "dice":
        return 2 * nab / (na + nb)
    if measure == "mi":
        return nab / (na * nb)
    raise ValueError("unknown measure: " + measure)

def top_associated(term, docs_sets, measure="dice", k=4):
    vocab = set().union(*docs_sets) - {term}
    scored = [(t, association(term, t, docs_sets, measure)) for t in vocab]
    scored = [(t, s) for t, s in scored if s > 0]
    return sorted(scored, key=lambda x: (-round(x[1], 12), x[0]))[:k]

# ---- Test (do not modify) ----
for term in ["cup", "world"]:
    for m in ["dice", "mi"]:
        print(f"{term:6} {m:5}: " + ", ".join(f"{t}({s:.3f})" for t, s in top_associated(term, docs_sets, m, 4)))

cup    dice : football(0.500), world(0.500), coffee(0.444), tea(0.444)
cup    mi   : champions(0.143), coffee(0.143), drink(0.143), fans(0.143)
world  dice : football(0.800), cup(0.500), fans(0.333), fifa(0.333)
world  mi   : fans(0.200), fifa(0.200), final(0.200), news(0.200)


In [34]:
# The output generated by the code written to answer this question should be as follows:

**Your answer (Exercise 5.1):**

- **Which measure favours rare terms?** **Mutual information** ($n_{ab}/(n_a n_b)$). The product $n_a n_b$ in the denominator punishes frequent terms, so terms that occur in only one document (champions, drink, fans, fifa, news…, each with $n_b=1$) get the highest scores and all tie (0.143 for *cup*, 0.200 for *world*), even though they are weakly supported by a single co-occurrence; the very related *football* is not even in the top 4. **Dice** normalises by $n_a + n_b$ and favours terms that frequently co-occur and are frequent enough, so *football* and *world* are top for *cup* (0.5) and *football* for *world* (0.8).
- **The *world cup* problem.** Associations are computed for each term separately: for *cup* the thesaurus proposes *coffee* and *tea* (from "a cup of coffee", "tea cup"), which are the wrong sense for the query *world cup*, while *world* proposes *football*/*news*. If each query term is expanded independently, the expansion for *cup* adds *coffee/tea* even though in the query context *world cup* means the football tournament, so the expanded query drifts off-topic and retrieves coffee-related documents. Term association measures are context-independent; expansion should take the whole query into account (e.g. pseudo-relevance feedback or query-level / contextual expansion).

### Exercise 5.2 — Explicit relevance feedback with Rocchio (7 points)

For the query **`"satellite launch"`** a user looked at the top results and marked **`d1`, `d3`** as relevant and **`d4`** as non-relevant. Apply the Rocchio algorithm (slide *"Rocchio Algorithm: in Practice"*):

$$\vec q_m = \alpha\,\vec q_0 + \beta\,\frac{1}{|D_r|}\sum_{\vec d_j\in D_r}\vec d_j - \gamma\,\frac{1}{|D_{nr}|}\sum_{\vec d_j\in D_{nr}}\vec d_j$$

with $\alpha = 1$, $\beta = 0.75$, $\gamma = 0.15$. Use the normalized `ltc` query vector for $\vec q_0$ and the normalized `lnc` document vectors from Part 2. **Set negative weights to 0** (standard practice).

1. Implement `centroid(vectors)` and `rocchio(q0, rel_vecs, nonrel_vecs, alpha, beta, gamma)`.
2. Print the 6 highest-weighted terms of $\vec q_m$.
3. Re-rank all documents by `cosine(normalize(q_m), d)` and print the ranking (excluding zeros).

In [35]:
def centroid(vectors):
    c = defaultdict(float)
    for v in vectors:
        for t, w in v.items():
            c[t] += w
    n = len(vectors)
    return {t: w / n for t, w in c.items()} if n else {}

def rocchio(q0, rel_vecs, nonrel_vecs, alpha=1.0, beta=0.75, gamma=0.15):
    qm = defaultdict(float)
    for t, w in q0.items():
        qm[t] += alpha * w
    for t, w in centroid(rel_vecs).items():
        qm[t] += beta * w
    for t, w in centroid(nonrel_vecs).items():
        qm[t] -= gamma * w
    return {t: w for t, w in qm.items() if w > 0}     # negative weights set to 0

# ---- Test (do not modify) ----
doc_vecs = {d: doc_vector_lnc(tf) for d, tf in TF.items()}
q0 = query_vector_ltc("satellite launch", DF, N)
qm = rocchio(q0, [doc_vecs["d1"], doc_vecs["d3"]], [doc_vecs["d4"]])
print("Top terms of q_m:", [(t, round(w, 4)) for t, w in sorted(qm.items(), key=lambda x: (-x[1], x[0]))[:6]])
qm_n = l2_normalize(qm)
ranking = sorted([(d, cosine(qm_n, v)) for d, v in doc_vecs.items()], key=lambda x: -x[1])
print("Before feedback:", [d for d, _ in rank_cosine("satellite launch", TF, DF, N)])
print("After Rocchio:  ")
for d, s in ranking:
    if s > 0:
        print(f"   {d:4} {s:.4f}")

Top terms of q_m: [('satellite', 1.0019), ('launch', 0.777), ('earth', 0.2948), ('orbit', 0.2948), ('images', 0.1531), ('scientists', 0.1531)]
Before feedback: ['d10', 'd1', 'd2', 'd4', 'd11', 'd3', 'd5']
After Rocchio:  
   d1   0.7637
   d10  0.6123
   d3   0.6055
   d2   0.5668
   d5   0.3990
   d4   0.2690
   d11  0.2513
   d8   0.1508
   d7   0.0870
   d18  0.0418


In [36]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 5.3 — Pseudo-relevance feedback (PRF) + evaluation (8 points)

This mirrors **Lab 6, Exercise 2**, but implemented from scratch on our small collection. The information need behind **`"satellite launch"`** is *"space missions: satellites, rockets and spacecraft"*, so the relevance judgments (qrels) are:

```
relevant = {d1, d2, d3, d5, d7, d8, d10, d16}      (all other documents are non-relevant)
```

Implement the four PRF steps:
1. Retrieve an initial ranking with **BM25** (Part 3, default parameters).
2. Assume the top **`k_docs`** documents are relevant; score each candidate term (not already in the query) by the **sum of its tf-idf weights** $(1+\log_{10}tf)\cdot\log_{10}(N/df)$ over those documents. Break ties alphabetically.
3. Expand the query with the top **`n_terms`** terms.
4. Re-run BM25 with the expanded query.

Write `score_feedback_terms(...)`, `expand_query(...)` and `prf_bm25(...)`. Run with `k_docs=3, n_terms=2`, print the expansion terms and both rankings, and evaluate both runs with **AP** and **P@5** from Part 1 (treat the ranking as the retrieved list).

In a markdown cell, discuss: did PRF help? Then try `(k_docs=2, n_terms=2)` and `(k_docs=5, n_terms=4)` and explain what you observe — in particular, where does `d11` (the bank) end up, and why is this called **query drift**?

In [37]:
qrels_sat = {"d1", "d2", "d3", "d5", "d7", "d8", "d10", "d16"}

def score_feedback_terms(feedback_docs, query_terms, TF, DF, N):
    scores = defaultdict(float)
    qset = set(query_terms)
    for d in feedback_docs:
        for t, c in TF[d].items():
            if t not in qset:
                scores[t] += log_tf(c) * idf(DF[t], N)
    return sorted(scores.items(), key=lambda x: (-round(x[1], 12), x[0]))

def expand_query(query, scored_terms, n_terms):
    return query + " " + " ".join(t for t, _ in scored_terms[:n_terms])

def prf_bm25(query, TF, DF, N, doc_len, k_docs=3, n_terms=2):
    initial = rank_bm25(query, TF, DF, N, doc_len)
    feedback_docs = [d for d, _ in initial[:k_docs]]
    scored = score_feedback_terms(feedback_docs, set(preprocess(query)), TF, DF, N)
    terms = [t for t, _ in scored[:n_terms]]
    expanded = expand_query(query, scored, n_terms)
    final = rank_bm25(expanded, TF, DF, N, doc_len)
    return terms, initial, final

# ---- Test (do not modify) ----
def evaluate(ranking, qrels):
    rels = [1 if d in qrels else 0 for d, _ in ranking]
    return average_precision(rels, len(qrels)), precision_at_k(rels + [0] * 5, 5)

terms, initial, final = prf_bm25("satellite launch", TF, DF, N, doc_len, k_docs=3, n_terms=2)
print("Expansion terms:", terms)
print("Initial :", [d for d, _ in initial])
print("Expanded:", [d for d, _ in final])
for name, run in [("BM25", initial), ("BM25+PRF", final)]:
    ap, p5 = evaluate(run, qrels_sat)
    print(f"{name:9} AP={ap:.4f}  P@5={p5:.4f}")

Expansion terms: ['rocket', 'orbit']
Initial : ['d10', 'd1', 'd2', 'd4', 'd11', 'd3', 'd5']
Expanded: ['d10', 'd2', 'd1', 'd3', 'd16', 'd4', 'd11', 'd7', 'd5', 'd8']
BM25      AP=0.5476  P@5=0.6000
BM25+PRF  AP=0.9160  P@5=1.0000


In [38]:
# The output generated by the code written to answer this question should be as follows:

In [39]:
# Extra runs requested in the exercise: other (k_docs, n_terms) settings
for kd, nt in [(3, 2), (2, 2), (5, 4)]:
    terms, initial, final = prf_bm25("satellite launch", TF, DF, N, doc_len, k_docs=kd, n_terms=nt)
    ap, p5 = evaluate(final, qrels_sat)
    ranked = [d for d, _ in final]
    print(f"k_docs={kd}, n_terms={nt}: terms={terms}")
    print(f"   ranking: {ranked}")
    print(f"   AP={ap:.4f}  P@5={p5:.4f}  rank of d11 = {ranked.index('d11') + 1 if 'd11' in ranked else None}")

k_docs=3, n_terms=2: terms=['rocket', 'orbit']
   ranking: ['d10', 'd2', 'd1', 'd3', 'd16', 'd4', 'd11', 'd7', 'd5', 'd8']
   AP=0.9160  P@5=1.0000  rank of d11 = 7
k_docs=2, n_terms=2: terms=['carried', 'nasa']
   ranking: ['d10', 'd1', 'd2', 'd4', 'd11', 'd3', 'd5']
   AP=0.5476  P@5=0.6000  rank of d11 = 5
k_docs=5, n_terms=4: terms=['rocket', 'orbit', 'account', 'bank']
   ranking: ['d11', 'd10', 'd2', 'd1', 'd3', 'd16', 'd4', 'd7', 'd5', 'd8']
   AP=0.7347  P@5=0.8000  rank of d11 = 1


**Your answer (Exercise 5.3):**

**Did PRF help?** Yes for the base setting (`k_docs=3, n_terms=2`). The top-3 BM25 documents (`d10`, `d1`, `d2`) are all relevant, so the expansion terms *rocket* and *orbit* are good; the expanded query retrieves additional relevant documents that contain no *satellite*/*launch* (`d16`, `d7`, `d8`) and AP rises from **0.5476 to 0.9160** and P@5 from **0.60 to 1.00** (the non-relevant `d4`/`d11` are pushed down: `d11` goes from rank 5 to rank 7, below relevant documents).

**Other settings (see the extra cell):**

| setting | expansion terms | AP | P@5 | rank of `d11` |
|---|---|---|---|---|
| k_docs=3, n_terms=2 | rocket, orbit | 0.9160 | 1.00 | 7 |
| k_docs=2, n_terms=2 | carried, nasa | 0.5476 | 0.60 | 5 |
| k_docs=5, n_terms=4 | rocket, orbit, account, bank | 0.7347 | 0.80 | **1** |

- With `k_docs=2` the feedback set (`d10`, `d1`) is too small: the top-scoring terms are *carried* and *nasa*, rare terms that appear in only one feedback document each (high idf, but they say nothing about the topic), so PRF brings no gain (same ranking as the baseline).
- With `k_docs=5, n_terms=4` the feedback set also contains `d4` and `d11` (the 4th and 5th BM25 results) – a phone launch and a *bank* launching a savings account – which are non-relevant. Their terms *account* and *bank* (rare, so high idf) enter the expansion, and `d11` becomes the **top result** (rank 1), AP drops to 0.7347 and P@5 to 0.80.
- This is **query drift**: PRF assumes that the top-k documents are relevant; when some of them are not, the expansion terms come from the wrong topic and the expanded query moves away from the original information need (here from space missions to banking), so non-relevant documents are promoted. PRF is therefore sensitive to the quality of the initial ranking, to `k_docs` (too few → noisy/rare terms; too many → includes non-relevant documents) and to `n_terms` (more terms → more chance of drift).